[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/14-what-the-model-learns.ipynb)

# Module 14: What the Model Learns

Companion notebook to [Module 14](https://transformers-and-math.drmhse.com/modules/14-what-the-model-learns) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0)

## Part 1 — Train a model that can only copy from its context (section 14.3)
Two attention-only layers, trained on random tokens that repeat with a **random period** between 6 and 15. The first period is unpredictable; after it, every token can be predicted by copying. Because the period changes from sequence to sequence, attending a fixed distance back does not work: the model has to find the earlier occurrence of the current token, which takes the two-layer induction circuit.

In [ ]:
V, T, D, H = 32, 32, 64, 2

def batch(B=64):
    seqs, periods = [], []
    for _ in range(B):
        p = int(torch.randint(6, 16, (1,)))
        first = torch.randperm(V)[:p]
        seqs.append(first.repeat(T // p + 2)[:T + 1])
        periods.append(p)
    s = torch.stack(seqs)
    return s[:, :-1], s[:, 1:], torch.tensor(periods)

class AttnOnly(nn.Module):
    def __init__(self, layers=2):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, D), nn.Embedding(T, D)
        self.layers = nn.ModuleList([nn.MultiheadAttention(D, H, batch_first=True) for _ in range(layers)])
        self.head = nn.Linear(D, V)
        self.ablate = None                                # (layer, head) to switch off
    def forward(self, ids):
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool), 1)
        x = self.tok(ids) + self.pos(torch.arange(T))
        attns, streams = [], [x]
        for l, layer in enumerate(self.layers):
            out, w = layer(x, x, x, attn_mask=mask, average_attn_weights=False)
            if self.ablate and self.ablate[0] == l:
                # recompute the layer's output with one head's weights zeroed
                h = self.ablate[1]
                w = w.clone(); w[:, h] = 0
                v = F.linear(x, layer.in_proj_weight[2 * D:], layer.in_proj_bias[2 * D:]).view(x.shape[0], T, H, D // H).transpose(1, 2)
                out = layer.out_proj((w @ v).transpose(1, 2).reshape(x.shape[0], T, D))
            x = x + out                                   # the residual stream (section 14.1)
            attns.append(w); streams.append(x)
        return self.head(x), attns, streams

def train(layers, steps=3000):
    torch.manual_seed(0)
    model = AttnOnly(layers)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for step in range(steps):
        x, y, _ = batch()
        loss = F.cross_entropy(model(x)[0].flatten(0, 1), y.flatten())
        opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def accuracy(model, n=512):
    torch.manual_seed(1)
    x, y, p = batch(n)
    correct = model(x)[0].argmax(-1) == y
    after = torch.arange(T)[None] >= p[:, None]          # positions after the first period
    return correct[after].float().mean().item(), correct[~after].float().mean().item()

model = train(2)
one_layer = train(1)
for name, m in [('two layers', model), ('one layer', one_layer)]:
    rep, first = accuracy(m)
    print(f'{name:10}: accuracy after the first period {rep:.1%}, within it {first:.1%} (chance {1 / V:.1%})')

### Find the heads
A previous-token head puts its weight on position $i - 1$. An induction head, at a token in a repeat of period $p$, puts its weight on position $i - p + 1$: the token that followed this one last time.

In [ ]:
@torch.no_grad()
def head_scores(model):
    torch.manual_seed(2)
    x, _, p = batch(256)
    _, attns, _ = model(x)
    for l, w in enumerate(attns):
        for h in range(H):
            prev = torch.stack([w[:, h, i, i - 1] for i in range(1, T)], 1).mean()
            ind = torch.stack([w[b, h, i, i - p[b] + 1] for b in range(len(p)) for i in range(int(p[b]), T)]).mean()
            print(f'layer {l + 1} head {h + 1}:  previous-token score {prev:.2f}   induction score {ind:.2f}')
    return x

x = head_scores(model)
with torch.no_grad():
    _, attns, _ = model(x[:1])
fig, axes = plt.subplots(2, H, figsize=(8, 8))
for l in range(2):
    for h in range(H):
        axes[l, h].imshow(attns[l][0, h], cmap='Reds'); axes[l, h].set_title(f'layer {l + 1}, head {h + 1}')
plt.tight_layout(); plt.show()

### Test the claim by switching heads off
The circuit has two parts, so removing either should hurt: the previous-token head in layer 1 (layer 2 can no longer tell which position followed which token) and the induction head(s) in layer 2. Removing an unrelated head should not. When two heads share the induction job, removing one only partly hurts.

In [ ]:
for l in range(2):
    for h in range(H):
        model.ablate = (l, h)
        print(f'without layer {l + 1} head {h + 1}: accuracy after the first period {accuracy(model)[0]:.1%}')
model.ablate = None

## Part 2 — The logit lens (section 14.4)
Read the prediction off the residual stream after each layer, using the final output layer.

In [ ]:
with torch.no_grad():
    torch.manual_seed(3)
    x, y, p = batch(256)
    _, _, streams = model(x)
    after = torch.arange(T)[None] >= p[:, None]
    for i, s in enumerate(streams):
        acc = (model.head(s).argmax(-1) == y)[after].float().mean()
        print(f'after {"embedding" if i == 0 else f"layer {i}":9}: accuracy after the first period {acc:.1%}')

## Part 3 — Superposition needs nearly perpendicular directions (section 14.5)

In [ ]:
for d in (64, 512, 4096):
    u = F.normalize(torch.randn(2000, d), dim=1)
    cos = (u @ u.T)[~torch.eye(2000, dtype=torch.bool)]
    print(f'd={d:5}: sd of the cosine between random directions = {cos.std():.4f}  (1/sqrt(d) = {d ** -0.5:.4f})')

## Your turn
1. Train with a *fixed* period of 16 instead. Does the two-layer circuit still form, or does a single head learn to look a fixed distance back? (Hint: a learned position table makes the shortcut possible.)
2. Why can one layer not implement an induction head (section 14.3)? What does the first layer have to write into the residual stream for the second to use?